# 1 · Who is who: matching the list of people to OpenAlex

OpenAlex identifies authors by **profiles** (ids such as `A5066437409`). The same name can belong to several people, and one person is sometimes split over several profiles. This notebook:

1. reads the list of people and shows what it understood;
2. downloads **all publications of the institution** over the period;
3. proposes, for each person, the matching OpenAlex profiles with a **confidence** level;
4. lets you confirm or reject the uncertain ones.

In [ ]:
# Run this cell first. On Google Colab it installs the package; locally it does nothing
# (install once with `pip install -e .` from the project folder).
import sys

if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/celoy/bibliometry"
    !pip install -q "git+{REPO_URL}"

In [ ]:
from bibliometry.env import project_root

ROOT = project_root()

In [ ]:
import pandas as pd

from bibliometry import pipeline
from bibliometry.config import Config
from bibliometry.roster import summarise_roster

pd.set_option("display.max_colwidth", 60)
config = Config.load(ROOT / "config.yaml")
config.validate()
print(f"{config.institution.name} ({config.institution.id}), {config.from_date} → {config.to_date}")

## 1. The list of people

Check the counts per sheet. If names, ORCIDs or labs are missing although they are in the file, the column was not recognised: rename it in the file (e.g. `Nom`, `Prénom`, `ORCID`, `Laboratoire`) or add a `columns:` mapping under `roster:` in `config.yaml`. The same lab written two ways (e.g. an old name) can be unified with `lab_aliases:` in `config.yaml`.

In [ ]:
roster = pipeline.load_roster(config)
summarise_roster(roster)

In [ ]:
roster.head(10)

## 2. All publications of the institution

Downloads every work with at least one author affiliated with the institution. A few thousand works take about a minute. If interrupted, run the cell again: it resumes.

In [ ]:
client = pipeline.make_client(config)
pipeline.harvest_institution_step(config, client)

## 3. Matching people to OpenAlex profiles

Candidates come from the ORCID in the list, from the authors of the institution's publications whose name fits, and — for people still without a candidate — from an OpenAlex name search.

* **high** confidence (same ORCID, or same full name with publications signed with the institution) is accepted automatically;
* **medium / low** wait for your verdict.

In [ ]:
matches, people = pipeline.match_step(config, client)
people.match_status.value_counts()

### To review

Each row is one candidate profile. Useful columns: `author_name`, `last_known_institution`, `top_topics`, `sample_titles`, and `openalex_url` (open it to see the profile).

In [ ]:
to_review = matches[~matches.accepted & (matches.verdict == "")]
to_review[["person_id", "author_id", "author_name", "confidence", "orcid_status",
           "works_in_period_with_institution", "last_known_institution", "top_topics",
           "sample_titles", "openalex_url"]]

### Give your verdicts

Either open `data/review/author_matches.csv` in Excel/LibreOffice and fill the `verdict` column with `ok` or `wrong`, **or** type them below. Then run the next cell to apply them (no new download). Verdicts are kept if you re-run this notebook.

In [ ]:
verdicts = {
    # ("person_id", "author_id"): "ok",      # this profile is the person
    # ("person_id", "author_id"): "wrong",   # it is somebody else
}
if verdicts:
    pipeline.set_verdicts(config, verdicts)

In [ ]:
matches, people = pipeline.match_step(config, None)  # None = no download, cached answers only
people.match_status.value_counts()

### People without a profile

Usually people who did not publish in the period (teaching staff, new PhD students), or whose name is written very differently in OpenAlex. If you know their OpenAlex id, add it in an `OpenAlex` column of the list and re-run.

In [ ]:
people[people.match_status == "not found"][["person_id", "lab", "orcid", "sheet"]]

Continue with `02_harvest_people.ipynb`.